# End-to-end preprocessing with the routed Document Slice pipeline

Same routing and summarisation as `ablation_dynamic_doc_slice_generation.ipynb`, but starting from the **PDFs** instead of the
frozen chunk JSONs, so that the whole preprocessing pipeline is timed in one go: Docling conversion + HybridChunker + cleaning,
then the slice-radius router, then the summariser. This is the *end-to-end scope* of the manuscript (the scope of the
71m34s / 29m09s notebook wall-clock figures of the conference paper), measured for the routed pipeline for the first time.

Outputs (new artefacts only, nothing frozen is overwritten):
- `split_documents_end_to_end/<pdf>.json` - the chunks of this run (same record format as `split_documents/`),
- `preprocessed_chunks/end_to_end_doc_slice_radius_dynamic.json` - chunks with routed radius and generated context,
- `benchmarking_results/preprocessing/end_to_end_routed_timings.json` - per-document and total wall-clock per stage,
- one `end_to_end_doc_slice_radius_dynamic` row in `emissions_data/emissions.csv` (codecarbon around the whole run).

Docling's CUDA cache is emptied after every conversion exactly as `doc_slice_chunking.ipynb` (the k=3 run) did.

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1"); os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")   # models are in the local cache
import hashlib, json, re, subprocess, time, unicodedata
from collections import Counter
import torch
import ollama
from codecarbon import EmissionsTracker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

# Tier 1 (deterministic table gate) + Tier 2 (single-forward-pass SLM) slice-radius predictor, unchanged.
from utils.dynamic_slice_prediction import DynamicSlicePredictor, OLLAMA_OPTIONS as PREDICTOR_OPTIONS

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
MAX_TOKENS = 2000
OLLAMA_MODEL_NAME = "gemma3:4b-it-qat"                        # ONE model for both the radius predictor and the summariser
SUMMARISER_MODELFILE = "Models/chunker_full_doc.Modelfile"    # SYSTEM + temperature 0 + num_ctx 16384, requested per call
SKIP_CONTEXT_FOR_K0 = True
DOC_LIMIT = None                                              # e.g. 2 for a smoke test; None = all PDFs in INPUT_DIR

RUN_NAME = "end_to_end_doc_slice_radius_dynamic"
INPUT_DIR = "input"                                           # the 25 scientific PDFs
CHUNK_DIR = "split_documents_end_to_end"                      # chunk JSONs of THIS run (split_documents/ stays frozen)
CHUNKS_WITH_METADATA_FILE_NAME = f"preprocessed_chunks/{RUN_NAME}.json"
TIMINGS_FILE = "benchmarking_results/preprocessing/end_to_end_routed_timings.json"
EMISSIONS_DIR = "./emissions_data"

_INT_PARAMS = {"num_ctx", "num_predict", "top_k", "seed", "num_gpu", "num_thread", "repeat_last_n", "num_batch", "num_keep", "mirostat"}

def load_modelfile(path):
    """(system_prompt, options) from an Ollama Modelfile (same helper as the ablation notebook)."""
    text = open(path, encoding="utf-8").read()
    m = re.search(r'^SYSTEM\s+"""(.*?)"""', text, flags=re.S | re.M)
    system = m.group(1) if m else None
    options = {}
    for name, value in re.findall(r"^PARAMETER\s+(\w+)\s+(.+?)\s*$", text, flags=re.M):
        value = value.strip().strip('"')
        if name in _INT_PARAMS:
            value = int(value)
        else:
            try: value = float(value)
            except ValueError: pass
        if name == "stop": options.setdefault("stop", []).append(value)
        else: options[name] = value
    return system, options

def clean_docling_chunk_strings(chunks):
    """Verbatim cleaning of doc_slice_chunking.ipynb / anthropic_traditional_chunking.ipynb."""
    cleaned = []
    for chunk in chunks:
        chunk = unicodedata.normalize("NFKD", chunk).replace(" ", " ")
        chunk = chunk.translate(str.maketrans({"–": "-", "—": "-", "‘": "'", "’": "'", "“": '"', "”": '"'}))
        chunk = re.sub(r"http\S+", "", chunk)
        chunk = re.sub(r"[ \t]+", " ", chunk)
        chunk = re.sub(r"\n\s*\n", "\n\n", chunk).strip()
        cleaned.append(chunk)
    return cleaned

SUMMARISER_SYSTEM_PROMPT, SUMMARISER_OPTIONS = load_modelfile(SUMMARISER_MODELFILE)
PREDICTOR_OPTIONS = {**PREDICTOR_OPTIONS, "num_ctx": SUMMARISER_OPTIONS["num_ctx"]}   # one resident runner for both phases
print(f"summariser = {OLLAMA_MODEL_NAME} + options {SUMMARISER_OPTIONS} + system prompt ({len(SUMMARISER_SYSTEM_PROMPT)} chars) | predictor options {PREDICTOR_OPTIONS}")

study_names = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".pdf"))
if DOC_LIMIT: study_names = study_names[:DOC_LIMIT]
os.makedirs(CHUNK_DIR, exist_ok=True); os.makedirs(EMISSIONS_DIR, exist_ok=True); os.makedirs(os.path.dirname(TIMINGS_FILE), exist_ok=True)
print(f"{len(study_names)} PDFs to process from {INPUT_DIR}/")

summariser = gemma3:4b-it-qat + options {'temperature': 0.0, 'num_ctx': 16384} + system prompt (789 chars) | predictor options {'temperature': 0.0, 'num_predict': 1, 'top_p': 1.0, 'num_ctx': 16384}
25 PDFs to process from input/


In [2]:
# ---- the whole preprocessing pipeline, timed per stage, inside one codecarbon tracker -------------------------------
converter = DocumentConverter()                                   # Docling defaults (layout model, table structure, RapidOCR), as in doc_slice_chunking.ipynb
tokenizer = HuggingFaceTokenizer(tokenizer=AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME), max_tokens=MAX_TOKENS)
chunker = HybridChunker(tokenizer=tokenizer, merge_peers=True)
predictor = DynamicSlicePredictor(model=OLLAMA_MODEL_NAME, options=PREDICTOR_OPTIONS)

tracker = EmissionsTracker(project_name=RUN_NAME, measure_power_secs=1, output_dir=EMISSIONS_DIR, log_level="error")

chunks_with_metadata = []
radius_counter = Counter()
summariser_loads = 0
per_doc = []                                                      # per-document wall-clock of the three stages
t_start = time.perf_counter()

with tracker:
    for source in study_names:
        # ---- Stage 1: PDF -> Docling document -> HybridChunker chunks -> cleaned strings -> chunk records (as in split_documents/)
        t0 = time.perf_counter()
        doc = converter.convert(os.path.join(INPUT_DIR, source)).document
        chunks_str = clean_docling_chunk_strings([c.text for c in chunker.chunk(dl_doc=doc)])
        chunks = [{"text": t, "document": source, "id": hashlib.sha256(t.encode()).hexdigest()} for t in chunks_str]
        with open(os.path.join(CHUNK_DIR, f"{source}.json"), "w", encoding="utf-8") as f:
            json.dump(chunks, f, indent=4, ensure_ascii=False)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()                              # free Docling's cache before the runner works, as the k=3 notebook did
        t1 = time.perf_counter()

        # ---- Stage 2: slice radius for every chunk (table gate without a call, otherwise one num_predict=1 forward pass)
        suggestions = [predictor.predict_with_details(c["text"])[:2] for c in chunks]
        radius_counter.update(k for k, _ in suggestions)
        t2 = time.perf_counter()

        # ---- Stage 3: summaries with the per-chunk radius (k = 0: no call, the chunk is embedded as is)
        for chunk_index, chunk in enumerate(chunks):
            doc_slice_radius, slice_tier = suggestions[chunk_index]
            if doc_slice_radius == 0 and SKIP_CONTEXT_FOR_K0:
                context = ""; text_to_embed = chunk["text"]
            else:
                start_index_original = chunk_index - doc_slice_radius
                start_index_truncated = max(0, start_index_original)
                end_index_original = chunk_index + doc_slice_radius
                end_index_truncated = min(len(chunks) - 1, end_index_original)
                if start_index_original < 0:
                    end_index_truncated = min(len(chunks) - 1, end_index_truncated + abs(start_index_original))
                if end_index_original > len(chunks) - 1:
                    start_index_truncated = max(0, start_index_truncated - abs(end_index_original - end_index_truncated))
                doc_slice = "FULL DOCUMENT:\n" + "".join(" " + chunks[i]["text"] for i in range(start_index_truncated, end_index_truncated + 1))
                history = [{"role": "user", "content": doc_slice}, {"role": "user", "content": f"CHUNK:\n{chunk['text']}"}]
                response = ollama.chat(model=OLLAMA_MODEL_NAME,
                                       messages=[{"role": "system", "content": SUMMARISER_SYSTEM_PROMPT}] + history,
                                       options=SUMMARISER_OPTIONS)
                summariser_loads += response["load_duration"] > 1e9
                context = response["message"]["content"]
                text_to_embed = context + "\n\n" + chunk["text"]
            chunks_with_metadata.append({"text": text_to_embed, "original_text": chunk["text"], "context": context, "document": source,
                                         "id": chunk["id"], "doc_slice_radius": doc_slice_radius, "slice_tier": slice_tier})
        t3 = time.perf_counter()
        per_doc.append({"document": source, "chunks": len(chunks), "parse_s": t1 - t0, "route_s": t2 - t1, "summarise_s": t3 - t2,
                        "radii": dict(Counter(k for k, _ in suggestions))})
        print(f"{source[:48]:48s} {len(chunks):3d} chunks | parse {t1-t0:6.1f} s | route {t2-t1:6.1f} s | summarise {t3-t2:6.1f} s")

    subprocess.run(["ollama", "stop", OLLAMA_MODEL_NAME], check=False)   # free the GPU once the whole run is over

elapsed = time.perf_counter() - t_start
totals = {k: sum(d[k] for d in per_doc) for k in ("parse_s", "route_s", "summarise_s")}
emissions_data = getattr(tracker, "final_emissions_data", None)
timings = {"run": RUN_NAME, "scope": "end-to-end preprocessing: Docling conversion + HybridChunker + cleaning, slice-radius routing, summarisation (no embedding)",
           "documents": len(per_doc), "chunks": sum(d["chunks"] for d in per_doc), "wall_clock_s": elapsed, "stages_s": totals,
           "radius_distribution": dict(sorted(radius_counter.items())), "summariser_reloads": summariser_loads,
           "codecarbon": None if emissions_data is None else {"timestamp": emissions_data.timestamp, "duration_s": emissions_data.duration,
                                                              "energy_kwh": emissions_data.energy_consumed, "emissions_kg": emissions_data.emissions},
           "ollama_version": ollama.Client()._client.get("/api/version").json()["version"], "per_document": per_doc}
with open(TIMINGS_FILE, "w", encoding="utf-8") as f:
    json.dump(timings, f, indent=2)
print(f"\nProcessed {len(per_doc)} PDFs, {timings['chunks']} chunks in {elapsed/60:.1f} min end to end "
      f"(parse {totals['parse_s']/60:.1f} + route {totals['route_s']/60:.1f} + summarise {totals['summarise_s']/60:.1f} min)")
print(f"Suggested radius distribution: {timings['radius_distribution']}  (k=0 chunks skipped the summariser: {SKIP_CONTEXT_FOR_K0}); summariser reloads: {summariser_loads}")
if emissions_data is not None:
    print(f"codecarbon: {emissions_data.emissions:.6f} kg CO2eq, {emissions_data.energy_consumed:.6f} kWh over {emissions_data.duration:.0f} s -> {EMISSIONS_DIR}/emissions.csv")
print(f"timings saved to {TIMINGS_FILE}")

[codecarbon WARNING @ 19:11:16] Multiple instances of codecarbon are allowed to run at the same time.


[INFO] 2026-10-05 19:11:17,839 [RapidOCR] base.py:22: Using engine_name: torch


[INFO] 2026-10-05 19:11:17,841 [RapidOCR] device_config.py:57: Using GPU device with ID: 0


[INFO] 2026-10-05 19:11:17,854 [RapidOCR] download_file.py:60: File exists and is valid: /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth


[INFO] 2026-10-05 19:11:17,855 [RapidOCR] main.py:50: Using /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth


...

srep03578.pdf                                     10 chunks | parse    2.5 s | route    1.9 s | summarise   10.2 s


srep04487.pdf                                     11 chunks | parse    5.1 s | route    1.9 s | summarise   10.7 s


srep05215.pdf                                     14 chunks | parse    5.0 s | route    1.7 s | summarise   19.0 s


RapidOCR returned empty result!


srep45325.pdf                                      9 chunks | parse    2.7 s | route    1.7 s | summarise   11.1 s

Processed 25 PDFs, 377 chunks in 10.5 min end to end (parse 1.7 + route 1.1 + summarise 7.7 min)
Suggested radius distribution: {0: 127, 2: 250}  (k=0 chunks skipped the summariser: True); summariser reloads: 0
codecarbon: 0.003199 kg CO2eq, 0.015242 kWh over 628 s -> ./emissions_data/emissions.csv
timings saved to benchmarking_results/preprocessing/end_to_end_routed_timings.json


In [3]:
# Save the processed chunks (new file; the frozen ablation artefacts are untouched) and compare this run's parsing with the frozen chunks.
with open(CHUNKS_WITH_METADATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)
print(f"Results saved to {CHUNKS_WITH_METADATA_FILE_NAME}")

frozen_ids = {c["id"] for n in os.listdir("split_documents") if n.endswith(".json") for c in json.load(open(os.path.join("split_documents", n), encoding="utf-8"))}
new_ids = {c["id"] for c in chunks_with_metadata}
print(f"chunk ids identical to the frozen split_documents/ set: {len(new_ids & frozen_ids)} of {len(new_ids)} new ({len(frozen_ids)} frozen)")

Results saved to preprocessed_chunks/end_to_end_doc_slice_radius_dynamic.json
chunk ids identical to the frozen split_documents/ set: 242 of 374 new (380 frozen)


Embedding and table creation are **not** part of the end-to-end preprocessing scope measured above (the conference figures did
not include them either). To build a LanceDB table from this run's chunks, reuse the last cell of
`ablation_dynamic_doc_slice_generation.ipynb` with `TABLE_NAME = RUN_NAME` and `CHUNKS_WITH_METADATA_FILE_NAME` as set here.